# 第二轮：1,140 字完整基线 · 训练 10 轮

“10层”在此按 **10 个 epoch（训练轮次）**设置；网络沿用首轮 ResNet18。
1,140 类随机初始化，使用已生成的完整训练集 218,880 张与 val_select 109,440 张。

**上传两个文件即可开始：**
1. 在“我的云端硬盘”中新建 `hanzi1140_round2`，上传原有 `hanzi1140_round2_data.zip`，保留文件名。
2. 在 Colab 上传本 Notebook，并保存到 Drive。选择 Python 3 和 GPU，先使用可用的 T4。
3. 从上到下运行六个代码单元：GPU → 数据准备 → 样例 → 小集短跑 → 完整 10 轮 → 曲线与开发报告。

训练代码已内嵌，无需另传脚本、字体或首轮模型。完整训练格默认一次完成 10 轮。
每完成一轮自动保存到 Drive；断线后重跑前面的准备格，再运行同一训练格即可恢复。


## ① 检查 GPU

使用 Colab 预装的 torch/torchvision，不升级这两个包。实际前向检查在子进程中完成。


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-u", "-c", "import sys\nimport torch, torchvision, numpy as np\nprint('Python:', sys.version)\nprint('PyTorch:', torch.__version__, 'Torchvision:', torchvision.__version__, 'NumPy:', np.__version__)\nif not torch.cuda.is_available():\n    raise RuntimeError('未分配 GPU：请更改运行时类型为 GPU，再重新连接。')\nprint('GPU:', torch.cuda.get_device_name(0))\nprint('显存 GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))\nwith torch.inference_mode():\n    model = torch.nn.Conv2d(1, 8, 3).cuda()\n    print('GPU 前向检查:', tuple(model(torch.zeros(2, 1, 64, 64, device='cuda')).shape))\n"], check=True)


## ② 连接 Drive，恢复固定代码与数据

在授权窗口完成 Drive 挂载。若文件夹名不同，修改本格中的 `DRIVE_ROOT`。
首次复制约 732 MiB 数据包到 `/content`，只解出训练所需的八个文件，校验后使用。
四集合的像素审计已在本机完成；此处检查文件指纹，不重新渲染或扫描全部像素。
源码快照保存到 `hanzi1140_round2/sources/`，代码和数据的标识随 run 保存。


In [ ]:
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/hanzi1140_round2')

DATA_ARCHIVE_SHA256 = 'de4a86f05d4d41420366dc3cf7de48271c9482167f9243727ee276d6a907494e'
DATA_MANIFEST_SHA256 = '6fde9718cdd5b4985d03b5a81f887f15c934ddda400775f93ba8c6f868dcdaef'
TRAINING_SOURCES = {'training/__init__.py': '"""81-character baseline training and reproducible data preparation."""\n', 'training/io_utils.py': '"""Small shared file helpers with no image-rendering dependency."""\nimport hashlib\nimport json\nfrom pathlib import Path\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef verify_dataset(output):\n    output = Path(output)\n    manifest = json.loads((output / "manifest.json").read_text(encoding="utf-8"))\n    for name, expected in manifest["files"].items():\n        if digest(output / name) != expected:\n            raise ValueError(f"Dataset file hash mismatch: {name}")\n    return manifest\n', 'training/train.py': '"""ResNet18 baseline, with epoch-boundary checkpoints and explicit test gating."""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport random\nimport time\n\nos.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Dataset\nimport torchvision\nfrom torchvision.models import resnet18\n\n# This module deliberately does not import the renderer: training consumes the\n# immutable, checked dataset and never regenerates examples or uses test labels.\nfrom training.io_utils import digest, write_json, verify_dataset\n\n\nclass GlyphDataset(Dataset):\n    def __init__(self, directory, split, per_class=None):\n        with np.load(Path(directory) / f"{split}.npz", allow_pickle=False) as data:\n            self.images = data["images"]\n            self.labels = data["labels"]\n            self.condition_ids = data["condition_ids"]\n        if per_class:\n            indexes = np.concatenate([np.flatnonzero(self.labels == y)[:per_class]\n                                      for y in np.unique(self.labels)])\n            self.images, self.labels = self.images[indexes], self.labels[indexes]\n            self.condition_ids = self.condition_ids[indexes]\n\n    def __len__(self):\n        return len(self.labels)\n\n    def __getitem__(self, index):\n        image = torch.from_numpy(self.images[index].copy()).unsqueeze(0).float().div_(127.5).sub_(1)\n        return image, int(self.labels[index]), index\n\n\ndef model_for(n_classes):\n    model = resnet18(weights=None)\n    model.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)\n    nn.init.kaiming_normal_(model.conv1.weight, mode="fan_out", nonlinearity="relu")\n    model.maxpool = nn.Identity()\n    model.fc = nn.Linear(model.fc.in_features, n_classes)\n    return model\n\n\ndef seed_everything(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    torch.use_deterministic_algorithms(True)\n\n\ndef environment(device):\n    return {"python": platform.python_version(), "torch": str(torch.__version__),\n            "torchvision": str(torchvision.__version__), "numpy": np.__version__,\n            "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),\n            "device": str(device),\n            "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None}\n\n\ndef save_checkpoint(path, state):\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    torch.save(state, temporary)\n    os.replace(temporary, path)\n\n\ndef loader_for(dataset, batch_size, device, seed=None):\n    # No online augmentation or worker RNG. A separate per-epoch generator makes\n    # the shuffle independent of evaluations and reproducible after resumption.\n    generator = torch.Generator().manual_seed(0 if seed is None else seed)\n    return DataLoader(dataset, batch_size=batch_size, shuffle=seed is not None,\n                      num_workers=0, pin_memory=device.type == "cuda", generator=generator)\n\n\ndef epoch_pass(model, loader, device, n_classes, optimizer=None, scaler=None):\n    training = optimizer is not None\n    model.train(training)\n    total_loss, n = 0.0, 0\n    matrix = np.zeros((n_classes, n_classes), dtype=np.int64)\n    all_predictions = np.empty(len(loader.dataset), dtype=np.int64)\n    with torch.set_grad_enabled(training):\n        for images, labels, indexes in loader:\n            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)\n            if training:\n                optimizer.zero_grad(set_to_none=True)\n            # Use AMP for training on CUDA; use fp32 for checkpoint selection.\n            with torch.autocast(device_type=device.type, enabled=training and device.type == "cuda"):\n                logits = model(images)\n                loss = nn.functional.cross_entropy(logits, labels)\n            if not torch.isfinite(loss):\n                raise FloatingPointError("Non-finite loss; checkpoint was not advanced")\n            if training:\n                scaler.scale(loss).backward()\n                scaler.step(optimizer)\n                scaler.update()\n            predicted = logits.detach().argmax(1).cpu().numpy()\n            truth = labels.cpu().numpy()\n            np.add.at(matrix, (truth, predicted), 1)\n            all_predictions[indexes.numpy()] = predicted\n            total_loss += loss.item() * len(labels)\n            n += len(labels)\n    denominator = matrix.sum(1)\n    metrics = {"loss": total_loss / n, "accuracy": float(np.trace(matrix) / n),\n               "macro_accuracy": float(np.mean(np.diag(matrix)[denominator > 0] / denominator[denominator > 0])),\n               "samples": n}\n    return metrics, matrix, all_predictions\n\n\ndef train(args, manifest, labels, device):\n    out = args.run_dir\n    out.mkdir(parents=True, exist_ok=True)\n    config = {\n        "mode": args.mode, "seed": args.seed, "epochs": args.epochs, "batch_size": args.batch_size,\n        "lr": args.lr, "weight_decay": args.weight_decay, "per_class": args.per_class,\n        "model": "resnet18-gray3x3-s1-no-maxpool-random", "loss": "cross_entropy",\n        "data_sha256": digest(args.data / "manifest.json"), "trainer_sha256": digest(__file__),\n        "io_sha256": digest(Path(__file__).with_name("io_utils.py")),\n        "environment": environment(device), "amp": device.type == "cuda",\n    }\n    if args.mode == "baseline" and args.per_class is not None:\n        raise ValueError("Per-class truncation is only allowed for smoke/overfit modes")\n    if not args.resume and any(out.iterdir()):\n        raise ValueError("Run directory is nonempty. Use --resume or choose a new run name.")\n    train_data = GlyphDataset(args.data, "train", args.per_class)\n    validation = train_data if args.mode == "overfit" else GlyphDataset(args.data, "val_select", args.per_class)\n    validation_loader = loader_for(validation, args.batch_size, device)\n    model = model_for(len(labels)).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs, eta_min=args.lr / 100)\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")\n    start, history, best_score = 0, [], (-math.inf, -math.inf)\n    if args.resume:\n        saved = torch.load(out / "last.pt", map_location="cpu", weights_only=True)\n        if saved["config"] != config:\n            raise ValueError("Resume configuration/environment/data/code changed. Restore the original settings or use a new run.")\n        model.load_state_dict(saved["model"])\n        optimizer.load_state_dict(saved["optimizer"])\n        scheduler.load_state_dict(saved["scheduler"])\n        scaler.load_state_dict(saved["scaler"])\n        torch.set_rng_state(saved["torch_rng"])\n        if device.type == "cuda":\n            torch.cuda.set_rng_state_all(saved["cuda_rng"])\n        start, history, best_score = saved["epoch"], saved["history"], tuple(saved["best_score"])\n        if not (out / "best.pt").exists():\n            raise ValueError("best.pt is missing; restore the complete run directory")\n        print(f"Resuming after completed epoch {start}", flush=True)\n    write_json(out / "config.json", config)\n    write_json(out / "history.json", history)\n    write_json(out / "labels.json", labels)\n    write_json(out / "data_manifest.json", manifest)\n    stop = min(args.epochs, start + args.stop_after) if args.stop_after else args.epochs\n    for epoch in range(start, stop):\n        beginning = time.perf_counter()\n        train_loader = loader_for(train_data, args.batch_size, device, seed=args.seed + epoch)\n        lr = optimizer.param_groups[0]["lr"]\n        train_metrics, _, _ = epoch_pass(model, train_loader, device, len(labels), optimizer, scaler)\n        val_metrics, _, _ = epoch_pass(model, validation_loader, device, len(labels))\n        scheduler.step()\n        record = {"epoch": epoch + 1, "lr": lr, "train": train_metrics,\n                  "validation_split": "train-subset" if args.mode == "overfit" else "val_select",\n                  "validation": val_metrics, "seconds_compute": time.perf_counter() - beginning}\n        history.append(record)\n        score = (val_metrics["macro_accuracy"], -val_metrics["loss"])\n        if score > best_score:\n            best_score = score\n            save_checkpoint(out / "best.pt", {"model": model.state_dict(), "config": config,\n                                               "epoch": epoch + 1, "metrics": val_metrics})\n        save_checkpoint(out / "last.pt", {\n            "model": model.state_dict(), "optimizer": optimizer.state_dict(),\n            "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),\n            "epoch": epoch + 1, "config": config, "history": history, "best_score": best_score,\n            "torch_rng": torch.get_rng_state(),\n            "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else [],\n        })\n        write_json(out / "history.json", history)\n        print(json.dumps(record, ensure_ascii=False), flush=True)\n        print(f"Checkpoint saved: {out / \'last.pt\'}; epoch wall time incl. saves: {time.perf_counter() - beginning:.1f}s", flush=True)\n    print(f"Completed {len(history)}/{args.epochs} epochs. Best macro accuracy: {best_score[0]:.4f}", flush=True)\n\n\ndef evaluate(args, manifest, label_data, device):\n    if args.split == "test" and not args.final_test:\n        raise ValueError("Test is reserved. Explicit --final-test is required after the protocol is frozen.")\n    checkpoint = torch.load(args.run_dir / "best.pt", map_location="cpu", weights_only=True)\n    if checkpoint["config"]["mode"] != "baseline":\n        raise ValueError("Only a baseline checkpoint can be used for development/final reports")\n    if checkpoint["config"]["data_sha256"] != digest(args.data / "manifest.json"):\n        raise ValueError("Checkpoint and dataset do not match")\n    if checkpoint["config"]["environment"] != environment(device):\n        raise ValueError("Evaluation environment differs from the training environment")\n    if checkpoint["config"]["trainer_sha256"] != digest(__file__):\n        raise ValueError("Evaluation code differs from the checkpoint source")\n    labels = label_data["characters"]\n    dataset = GlyphDataset(args.data, args.split)\n    model = model_for(len(labels)).to(device)\n    model.load_state_dict(checkpoint["model"])\n    metrics, matrix, prediction = epoch_pass(model, loader_for(dataset, args.batch_size, device), device, len(labels))\n    conditions = json.loads((args.data / "conditions.json").read_text(encoding="utf-8"))[args.split]\n    sigma = np.array([conditions[int(i)]["sigma"] for i in dataset.condition_ids])\n    metrics["by_sigma"] = {str(s): {"samples": int((sigma == s).sum()),\n        "accuracy": float(np.mean(prediction[sigma == s] == dataset.labels[sigma == s]))}\n        for s in sorted(set(sigma.tolist()))}\n    metrics.update({"split": args.split, "checkpoint_epoch": checkpoint["epoch"],\n                    "checkpoint_sha256": digest(args.run_dir / "best.pt")})\n    char_to_id = {c["char"]: c["class_id"] for c in labels}\n    pairs = []\n    for pair in label_data["pairs"]:\n        a, b = char_to_id[pair["a"]], char_to_id[pair["b"]]\n        pairs.append({"a": pair["a"], "b": pair["b"], "a_to_b": int(matrix[a, b]),\n                      "b_to_a": int(matrix[b, a]),\n                      "pair_error_rate": float((matrix[a, b] + matrix[b, a]) / (matrix[a].sum() + matrix[b].sum()))})\n    report_dir = args.run_dir / f"evaluation_{args.split}"\n    report_dir.mkdir(exist_ok=True)\n    np.save(report_dir / "confusion.npy", matrix)\n    write_json(report_dir / "metrics.json", metrics)\n    write_json(report_dir / "pairs.json", pairs)\n    with (report_dir / "predictions.csv").open("w", newline="", encoding="utf-8-sig") as stream:\n        writer = csv.writer(stream)\n        writer.writerow(["index", "true_id", "true_char", "predicted_id", "predicted_char", "sigma", "dx", "dy"])\n        for index, (truth, pred) in enumerate(zip(dataset.labels, prediction)):\n            condition = conditions[int(dataset.condition_ids[index])]\n            writer.writerow([index, truth, labels[int(truth)]["char"], pred, labels[int(pred)]["char"],\n                             condition["sigma"], condition["dx"], condition["dy"]])\n    print(json.dumps(metrics, ensure_ascii=False, indent=2))\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("command", choices=["train", "evaluate"])\n    parser.add_argument("--data", type=Path, required=True)\n    parser.add_argument("--run-dir", type=Path, required=True)\n    parser.add_argument("--mode", choices=["smoke", "overfit", "baseline"], default="baseline")\n    parser.add_argument("--epochs", type=int, default=40)\n    parser.add_argument("--batch-size", type=int, default=64)\n    parser.add_argument("--lr", type=float, default=0.001)\n    parser.add_argument("--weight-decay", type=float, default=0.0001)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--per-class", type=int)\n    parser.add_argument("--stop-after", type=int, help="Stop after N additional epochs without changing the planned schedule")\n    parser.add_argument("--resume", action="store_true")\n    parser.add_argument("--allow-cpu", action="store_true", help="Only for local verification")\n    parser.add_argument("--split", choices=["val_select", "val_graph", "test"], default="val_select")\n    parser.add_argument("--final-test", action="store_true")\n    args = parser.parse_args()\n    if min(args.epochs, args.batch_size) <= 0 or (args.per_class is not None and args.per_class <= 0):\n        parser.error("epochs, batch-size and per-class must be positive")\n    if args.stop_after is not None and args.stop_after <= 0:\n        parser.error("stop-after must be positive")\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    if device.type == "cpu" and not args.allow_cpu:\n        raise RuntimeError("No CUDA GPU. In Colab select Runtime > Change runtime type > GPU, then reconnect.")\n    if device.type == "cpu":\n        torch.set_num_threads(2)\n    seed_everything(args.seed)\n    manifest = verify_dataset(args.data)\n    label_data = json.loads((args.data / "labels.json").read_text(encoding="utf-8"))\n    print(json.dumps(environment(device), indent=2), flush=True)\n    if args.command == "train":\n        train(args, manifest, label_data["characters"], device)\n    else:\n        evaluate(args, manifest, label_data, device)\n\n\nif __name__ == "__main__":\n    main()\n'}

"""Second-round Colab setup; source text and fixed data hashes are embedded by the generator."""
from pathlib import Path
import hashlib
import json
import shutil
import sys
import zipfile

LOCAL_ROOT = Path('/content')
ARCHIVE_NAME = 'hanzi1140_round2_data.zip'
DATA_MEMBERS = ('manifest.json', 'audit.json', 'labels.json', 'conditions.json',
                'train.npz', 'val_graph.npz', 'val_select.npz', 'test.npz')


def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def prepare_round2(root, local_root, archive_hash, manifest_hash, source_texts):
    root, local_root = Path(root), Path(local_root)
    source_id = hashlib.sha256(json.dumps(source_texts, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
    work = local_root / f'hanzi1140-{archive_hash[:12]}-{source_id[:12]}'
    data = work / 'training/data/round2-1140-v1'
    data.mkdir(parents=True, exist_ok=True)
    archive_path = root / ARCHIVE_NAME
    if not archive_path.is_file():
        raise FileNotFoundError(f'请先将数据包上传到：{archive_path}')
    # Read a local copy during training, retaining the uploaded immutable ZIP on Drive.
    local_zip = local_root / ARCHIVE_NAME
    if not local_zip.is_file() or file_sha256(local_zip) != archive_hash:
        temporary = local_zip.with_suffix('.zip.copying')
        shutil.copyfile(archive_path, temporary)
        if file_sha256(temporary) != archive_hash:
            raise ValueError('数据包 SHA-256 不匹配；请上传本轮原始数据包。')
        temporary.replace(local_zip)
    with zipfile.ZipFile(local_zip) as bundle:
        # Explicit names only: PNG previews and other ZIP entries are not extracted.
        for name in DATA_MEMBERS:
            destination = data / name
            temporary = destination.with_suffix(destination.suffix + '.copying')
            with bundle.open('round2-1140-v1/' + name) as source, temporary.open('wb') as target:
                shutil.copyfileobj(source, target)
            temporary.replace(destination)
    if file_sha256(data / 'manifest.json') != manifest_hash:
        raise ValueError('数据 manifest 与冻结的第二轮版本不一致。')
    manifest = json.loads((data / 'manifest.json').read_text(encoding='utf-8'))
    for name, expected in manifest['files'].items():
        if file_sha256(data / name) != expected:
            raise ValueError(f'数据文件校验失败：{name}')
    counts = {name: value['images'] for name, value in manifest['splits'].items()}
    if counts != {'train': 218880, 'val_graph': 109440, 'val_select': 109440, 'test': 218880}:
        raise ValueError('不是本轮完整 1,140 字数据。')
    label_data = json.loads((data / 'labels.json').read_text(encoding='utf-8'))
    if [r['class_id'] for r in label_data['characters']] != list(range(1140)):
        raise ValueError('类别映射不等于 0～1139。')
    audit = json.loads((data / 'audit.json').read_text(encoding='utf-8'))
    if not audit['automatic_checks_passed'] or audit['unique_images'] != 656640:
        raise ValueError('缺少本轮已完成的像素审计。')
    snapshots = root / 'sources' / source_id
    for name, source in source_texts.items():
        encoded = source.encode('utf-8')
        compile(source, name, 'exec')
        target = work / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(encoded)
        snapshot = snapshots / name
        snapshot.parent.mkdir(parents=True, exist_ok=True)
        if snapshot.exists():
            if snapshot.read_bytes() != encoded:
                raise ValueError(f'Drive 中的源码快照已变化：{snapshot}')
        else:
            snapshot.write_bytes(encoded)
    (root / 'runs').mkdir(exist_ok=True)
    print('数据文件指纹与既有离线审计通过：', counts)
    print('代码和数据位于：', work)
    print('检查点保存到：', root / 'runs')
    return work, data, source_id


if __name__ == '__main__':
    from google.colab import drive
    drive.mount('/content/drive')
    WORK, DATA, SOURCE_ID = prepare_round2(
        DRIVE_ROOT, LOCAL_ROOT, DATA_ARCHIVE_SHA256, DATA_MANIFEST_SHA256, TRAINING_SOURCES)


## ③ 检查训练图片

从全字表均匀选择 16 个类别，显示实际训练图。字符对照在图下方输出。


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

labels = json.loads((DATA / 'labels.json').read_text(encoding='utf-8'))['characters']
indexes = np.linspace(0, 1139, 16, dtype=int)
with np.load(DATA / 'train.npz', allow_pickle=False) as saved:
    # Decode the large image array once, then retain only the 16 preview tiles.
    tiles = saved['images'][indexes]
    targets = saved['labels'][indexes]
fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for image, target, ax in zip(tiles, targets, axes.flat):
    ax.imshow(image, cmap='gray', vmin=0, vmax=255, interpolation='nearest')
    ax.set_title(f'ID {int(target)}')
    ax.axis('off')
fig.tight_layout()
fig.savefig(DRIVE_ROOT / 'sample_preview.png', dpi=150)
plt.show()
plt.close(fig)
print(' / '.join(f"{int(i)}: {labels[int(i)]['char']}" for i in targets))
del tiles, targets


## ④ 一轮小集短跑

每类选 2 张训练图和 2 张 val_select 图，检查 1,140 类输出、梯度与 Drive 保存。
这个独立 run 训练 1 轮，检查通过后下一格新建完整基线；完整基线的预算为 10 轮。
再次运行会复用已完成的短跑检查点。此处准确率只用于流程检查。


In [ ]:
"""Shared notebook helpers using the unchanged first-round training entry point."""
from pathlib import Path
import hashlib
import json
import subprocess
import sys

TOTAL_EPOCHS = 10
BATCH_SIZE = 64
SEED = 42
RUN_NAME = 'baseline1140_e10_s42_v1'


def checkpoint_digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def run_round2_training(work, data, root, name, mode, epochs, per_class=None, stop_after=None):
    folder = Path(root) / 'runs' / name
    command = [sys.executable, '-u', '-m', 'training.train', 'train',
        '--data', str(data), '--run-dir', str(folder), '--mode', mode,
        '--epochs', str(epochs), '--batch-size', str(BATCH_SIZE), '--seed', str(SEED),
        '--lr', '0.001', '--weight-decay', '0.0001']
    if per_class is not None:
        command += ['--per-class', str(per_class)]
    if stop_after is not None:
        command += ['--stop-after', str(stop_after)]
    if (folder / 'last.pt').exists():
        command += ['--resume']
    subprocess.run(command, cwd=work, check=True)
    for name in ('config.json', 'history.json', 'best.pt', 'last.pt'):
        if not (folder / name).is_file():
            raise RuntimeError(f'训练输出缺失：{folder / name}')
    provenance = {'data_archive_sha256': DATA_ARCHIVE_SHA256, 'training_source_id': SOURCE_ID}
    (folder / 'notebook_provenance.json').write_text(
        json.dumps(provenance, indent=2) + '\n', encoding='utf-8')
    environment = folder / 'environment-pip-freeze.txt'
    if not environment.exists():
        freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], check=True, capture_output=True, text=True)
        environment.write_text(freeze.stdout, encoding='utf-8')
    print('已保存：', folder)
    return folder


def evaluate_completed_round2(work, data, folder):
    folder = Path(folder)
    history = json.loads((folder / 'history.json').read_text(encoding='utf-8'))
    if [r['epoch'] for r in history] != list(range(1, TOTAL_EPOCHS + 1)):
        print(f'当前 {len(history)}/{TOTAL_EPOCHS} 轮；完整训练后才生成开发集混淆报告。')
        return []
    best_hash = checkpoint_digest(folder / 'best.pt')
    marker = folder / 'development_complete.json'
    splits = ('val_select', 'val_graph')
    if marker.exists() and json.loads(marker.read_text())['checkpoint_sha256'] == best_hash:
        required = [folder / f'evaluation_{split}' / name for split in splits
                    for name in ('metrics.json', 'confusion.npy', 'pairs.json', 'predictions.csv')]
        if all(path.is_file() for path in required):
            print('复用已保存的两个开发集报告。')
            return list(splits)
    for split in splits:
        subprocess.run([sys.executable, '-u', '-m', 'training.train', 'evaluate',
            '--data', str(data), '--run-dir', str(folder), '--split', split,
            '--batch-size', str(BATCH_SIZE)], cwd=work, check=True)
    temporary = marker.with_suffix('.json.tmp')
    temporary.write_text(json.dumps({'checkpoint_sha256': best_hash, 'splits': list(splits)}, indent=2) + '\n')
    temporary.replace(marker)
    return list(splits)


smoke_dir = run_round2_training(WORK, DATA, DRIVE_ROOT, RUN_NAME + '_smoke', 'smoke', 1, per_class=2)


## ⑤ 完整训练 10 轮

运行本格会开始训练。模型、损失与优化设置沿用首轮：灰度 ResNet18、3×3 stride=1 输入、
无首层 maxpool、随机初始化、交叉熵、AdamW、batch=64、lr=0.001、weight_decay=0.0001、seed=42。
余弦学习率按 **总计 10 轮**下降到初始值的 1/100；best.pt 按 val_select macro accuracy 最高选择，
并列取交叉熵更低者。训练采用 CUDA AMP，验证采用 fp32。

`EPOCHS_THIS_SESSION=None` 一次完成全部轮次；设为 2 时，本次最多追加两轮，总预算仍为 10。
断线后重新准备数据并运行此格，会从 `last.pt` 的最近完整轮次恢复。
保持原 GPU 型号、软件版本和超参数；需要改 batch 等设置时，改第④格的 `RUN_NAME` 新建 run。
同一 run 只在一个 Colab 会话中写入。


In [ ]:
EPOCHS_THIS_SESSION = None  # None=本次完成剩余全部轮次；2=本次最多追加两轮
baseline_dir = run_round2_training(
    WORK, DATA, DRIVE_ROOT, RUN_NAME, 'baseline', TOTAL_EPOCHS,
    stop_after=EPOCHS_THIS_SESSION)


## ⑥ 曲线与完整开发报告

先保存损失／准确率曲线；完成 10 轮后，用 best.pt 分别评估 val_select 与 val_graph。
输出总体和 sigma 分项准确率、1,140×1,140 混淆矩阵、首轮保留的 54 对误认及逐图预测。
新增字组保存在标签表中，多字组未自动展开为全部两两关系。
报告会列出错误最多的类别，便于下一步判断扩类后的错误空间；10 轮完成本身不代表已经收敛。

结果目录：`MyDrive/hanzi1140_round2/runs/baseline1140_e10_s42_v1/`。
`best.pt` 是所选模型，`last.pt` 保存完整恢复状态，`history.json` 和 `config.json` 保存训练记录。
曲线为 `learning_curves.png`，报告在 `evaluation_val_select/` 与 `evaluation_val_graph/`，
汇总为 `notebook_summary.json`。同一检查点的完整开发报告会复用。
test 留待最终评价，本 Notebook 仅执行两个开发集推理。
确认 Drive 中保存完成后，在菜单中断开连接并删除运行时。


In [ ]:
"""Plot saved history and generate the two development reports after all ten epochs."""
import json
import numpy as np
import matplotlib.pyplot as plt

baseline_dir = DRIVE_ROOT / 'runs' / RUN_NAME
history = json.loads((baseline_dir / 'history.json').read_text(encoding='utf-8'))
if not history:
    raise ValueError('尚无已完成的训练轮次。请先运行训练格。')
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for key, caption in (('train', 'train'), ('validation', 'val_select')):
    epochs = [r['epoch'] for r in history]
    axes[0].plot(epochs, [r[key]['loss'] for r in history], marker='.', label=caption)
    axes[1].plot(epochs, [r[key]['accuracy'] for r in history], marker='.', label=caption)
axes[0].set_ylabel('cross entropy')
axes[1].set_ylabel('accuracy')
for ax in axes:
    ax.set_xlabel('epoch')
    ax.grid(alpha=0.3)
    ax.legend()
fig.tight_layout()
fig.savefig(baseline_dir / 'learning_curves.png', dpi=150)
plt.show()
plt.close(fig)

completed_splits = evaluate_completed_round2(WORK, DATA, baseline_dir)
summary = {'completed_epochs': len(history), 'planned_epochs': TOTAL_EPOCHS,
           'run': RUN_NAME, 'data_archive_sha256': DATA_ARCHIVE_SHA256,
           'training_source_id': SOURCE_ID,
           'seconds_compute_train_and_validation': sum(r['seconds_compute'] for r in history),
           'development_reports': {}}
label_rows = json.loads((DATA / 'labels.json').read_text(encoding='utf-8'))['characters']
for split in completed_splits:
    directory = baseline_dir / f'evaluation_{split}'
    metrics = json.loads((directory / 'metrics.json').read_text(encoding='utf-8'))
    summary['development_reports'][split] = metrics
    print(split, json.dumps(metrics, ensure_ascii=False, indent=2))
    matrix = np.load(directory / 'confusion.npy', allow_pickle=False)
    if matrix.shape != (1140, 1140) or int(matrix.sum()) != 109440:
        raise ValueError(f'{split} 混淆矩阵规模不正确。')
    off_diagonal = matrix.copy()
    np.fill_diagonal(off_diagonal, 0)
    errors = off_diagonal.sum(axis=1)
    indexes = np.argsort(-errors)[:20]
    print('错误最多的类别（真实字 / 错误数 / 最常误认字 / 该方向次数）：')
    for source in indexes:
        if not errors[source]:
            break
        target = int(off_diagonal[source].argmax())
        print(label_rows[source]['char'], int(errors[source]), label_rows[target]['char'], int(off_diagonal[source, target]))
    summary['development_reports'][split]['sources_with_top_confusion_ge5'] = int((off_diagonal.max(axis=1) >= 5).sum())
(baseline_dir / 'notebook_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(f'已完成 {len(history)}/{TOTAL_EPOCHS} 轮。完整曲线与报告：{baseline_dir}')
print('确认 Drive 中保存完成后，断开连接并删除运行时。')
